# Mon environnement de travail Python/AI

Ce notebook documente la configuration de mon environnement de travail.  
Il sert de référence personnelle et de point de départ pour tout le reste.

**Date de création :** 2026-03-27 (GX10) — étendu pour Windows 2026-05  
**Machines couvertes :**

- **Asus Ascent GX10** — ARM64, Ubuntu, calcul AI principal, accès SSH, JupyterLab sur port 11002
- **Laptop Windows 11** — x64, développement et notebooks via VSCode (extension Jupyter)

Quand une commande ou un chemin diffère entre les deux plateformes, le notebook indique explicitement laquelle est concernée.

## 1. Pourquoi conda ?

### Le problème à résoudre

En Python (et particulièrement en AI/ML), un projet dépend de versions très précises de paquets : `numpy 1.24`, `pytorch 2.1`, `tensorflow 2.15`, etc. Deux projets peuvent demander des versions incompatibles du même paquet. Installer tout dans le Python système crée vite un cocktail instable où mettre à jour un paquet casse un autre projet.

**Analogie C++ :** c'est l'équivalent d'avoir une seule version d'une lib (genre Boost) installée system-wide alors que tes projets demandent Boost 1.70, 1.75 et 1.82 simultanément. Soit tu fais cohabiter via des chemins isolés, soit tout pète.

### Ce que fait conda

Conda est un **gestionnaire d'environnements et de paquets**. Pour chaque projet, tu crées un *environnement* — un dossier autonome qui contient :

- son propre interpréteur Python (avec sa version exacte)
- ses propres paquets installés
- ses propres binaires et bibliothèques natives (DLL Windows, `.so` Linux)

Aucun croisement entre environnements. C'est l'isolation par dossier — équivalent conceptuel d'un chroot léger sous Linux, ou d'un assembly side-by-side sous Windows.

### Conda vs venv vs pip

| Outil | Ce qu'il gère | Ce qu'il NE gère pas |
|---|---|---|
| `pip` | Paquets Python (depuis PyPI) | Version de Python, libs natives non-Python |
| `venv` | Environnement Python (dossier isolé) | Version de Python (utilise celui du système), libs natives |
| `conda` | Paquets + Python + libs natives + non-Python (R, CUDA, etc.) | Rien dans ce périmètre |

**Pourquoi conda gagne pour l'AI :** beaucoup de paquets ML (PyTorch avec CUDA, TensorFlow GPU, OpenCV) ont des dépendances binaires natives. `pip` les installe parfois mais avec des galères de compatibilité ABI. `conda` (via les channels `conda-forge` ou `pytorch`) livre des binaires précompilés cohérents avec ton OS et ton CUDA — c'est précisément l'ABI qui est testée et garantie par le channel.

### Pourquoi un kernel Jupyter par environnement ?

JupyterLab est un *serveur* qui tourne avec son propre Python. Mais chaque notebook peut s'exécuter sur **un Python différent** grâce aux **kernels**. Le paquet `ipykernel` (installé dans chaque env conda) crée le pont entre l'env et JupyterLab.

Conséquence pratique : tu peux avoir un seul JupyterLab qui sert plusieurs notebooks, chacun tournant dans son propre env conda. C'est l'analogue d'un IDE qui peut charger plusieurs projets, chacun avec son runtime distinct.

## 2. Architecture conda sur ma machine

### Où vivent les choses

| Élément | GX10 (Ubuntu) | Windows |
|---|---|---|
| Installation Miniconda | `/home/bbrisson/miniconda3/` | `C:\Users\benoi\miniconda3\` |
| Environnement `base` | (mêmes chemins ci-dessus) | (mêmes chemins ci-dessus) |
| Environnements *named* | `/home/bbrisson/miniconda3/envs/` | `C:\Users\benoi\.conda\envs\` |
| Cache de paquets téléchargés | `~/miniconda3/pkgs/` | `C:\Users\benoi\miniconda3\pkgs\` |
| Config utilisateur | `~/.condarc` | `C:\Users\benoi\.condarc` |

**Note Windows :** Miniconda installé en mode utilisateur place les envs *named* dans `C:\Users\benoi\.conda\envs\`, **pas** dans `miniconda3\envs\`. C'est une particularité du mode user-install — pour ne pas réclamer les droits admin pour créer un env.

### Le concept d'environnement `base`

`base` est l'env créé automatiquement à l'installation de Miniconda. C'est l'env "racine" qui contient les outils de conda lui-même (`conda`, `pip`, parfois `python`). Quand ton prompt affiche `(base)`, c'est que tu es dans cet env par défaut.

**Règle d'or :** ne jamais installer de paquets de projet dans `base`. Garder `base` minimal. Créer un env dédié par projet pour ne jamais corrompre l'outil conda lui-même. Si `base` est cassé, plus rien ne marche.

### Named envs vs prefix envs

Conda distingue **deux façons** de créer un environnement :

**1. Named env** — créé avec `conda create -n nom_env` :

- Stocké dans un emplacement standard (`envs_dirs` configuré dans `~/.condarc`)
- Activé par nom court : `conda activate nom_env`
- C'est ce qu'on utilise pour les projets long terme

**2. Prefix env** — créé avec `conda create -p /chemin/absolu` :

- Stocké où tu veux, typiquement à côté du projet
- Activé par chemin complet : `conda activate C:\chemin\complet`
- Pas de nom court — apparaît "sans nom" dans `conda env list`

**Pourquoi cette distinction existe ?** Les prefix envs servent aux outils qui veulent un env auto-contenu, indépendant de ta config conda perso (souvent des installateurs tiers genre `roop-unleashed`, `comfyui`, etc.). Quand tu vois plein de lignes sans nom dans `conda env list`, ce sont des prefix envs créés par ces outils.

**Analogie LabVIEW :** les prefix envs, c'est comme un projet LabVIEW qui livre son propre Run-Time Engine dans son dossier, pour ne dépendre de rien d'externe.

In [ ]:
# Lister tous les envs conda visibles depuis le kernel actuel.
# Windows : marche directement (conda est dans le PATH de cmd.exe).
# GX10   : il faut le chemin absolu — remplacer par
#          !/home/bbrisson/miniconda3/bin/conda env list
!conda env list

## 3. `environment.yml` : le manifeste du projet

Le fichier [environment.yml](../environment.yml) à la racine du repo est l'équivalent d'un `requirements.txt` (Python pur), d'un `package.json` (Node) ou d'un `vcpkg.json` (C++) : il déclare **quel environnement le projet attend**.

Contenu typique :

```yaml
name: ai_learning
channels:
  - conda-forge
dependencies:
  - python=3.11
  - ipykernel
  - numpy
  - pandas
```

### À quoi ça sert concrètement

| Action | Commande |
|---|---|
| Recréer l'env sur une autre machine | `conda env create -f environment.yml` |
| Mettre à jour l'env après modification du fichier | `conda env update -f environment.yml --prune` |
| Exporter l'état actuel d'un env vers un fichier | `conda env export --from-history > environment.yml` |

**Le flag `--from-history` est important** : sans lui, l'export inclut **tous** les paquets installés y compris les dépendances transitives avec versions précises et souvent des hash spécifiques à la plateforme (`numpy=1.24.3=py311h...`). Avec `--from-history`, seuls les paquets que tu as **explicitement demandés** apparaissent (`numpy`) — c'est portable entre Linux/Windows/macOS.

**Analogie git :** sans `--from-history`, l'export est comme un lockfile (tout fixé en versions exactes). Avec, c'est comme un manifeste minimal (contraintes principales seulement).

### Pourquoi le `prefix:` a été retiré

Quand on exporte sans `--from-history`, conda ajoute parfois une ligne `prefix: /home/bbrisson/miniconda3/envs/ai_learning` à la fin du fichier. Cette ligne est **ignorée** par `conda env create` (elle est purement informative), mais elle n'a aucune valeur portable et trahit la machine d'origine. On l'a supprimée pour clarté et pour que le fichier soit utilisable sur les deux plateformes sans modification.

### Workflow recommandé

1. Tu veux ajouter une dépendance → édite `environment.yml` à la main pour ajouter la ligne
2. Applique : `conda env update -f environment.yml --prune`
3. Commit le fichier modifié → ton autre machine recevra la mise à jour au prochain `git pull`

## 4. Référence pratique : commandes conda

### Inspection

| Commande | Ce qu'elle affiche |
|---|---|
| `conda --version` | Version de conda installée |
| `conda info` | Détails complets : env actif, chemins, channels, plateforme |
| `conda env list` | Tous les envs ; `*` à côté de l'actif |
| `conda list -n ai_learning` | Tous les paquets installés dans l'env (sans l'activer) |
| `conda list -n ai_learning numpy` | Filtre : juste numpy |
| `conda search numpy` | Versions de numpy disponibles dans les channels configurés |

### Gestion d'environnements

| Commande | Effet |
|---|---|
| `conda create -n mon_env python=3.11 -y` | Crée un env named avec Python 3.11 |
| `conda env create -f environment.yml` | Crée depuis un manifeste (voir section 3) |
| `conda activate mon_env` | Active l'env (prompt change) |
| `conda deactivate` | Revient à l'env précédent (souvent `base`) |
| `conda remove -n mon_env --all` | Supprime complètement l'env |
| `conda rename -n vieux nouveau` | Renomme un env |

### Installation de paquets

| Commande | Effet |
|---|---|
| `conda install -n mon_env numpy` | Installe numpy dans `mon_env` (sans l'activer) |
| `conda install numpy` | Installe dans l'env **actif** |
| `conda install -c conda-forge polars` | Installe depuis le channel `conda-forge` |
| `conda update numpy` | Met à jour numpy |
| `conda update --all` | Met à jour tous les paquets de l'env actif |

### Exécution ponctuelle dans un env sans l'activer

```bash
conda run -n ai_learning python script.py
conda run -n ai_learning python -c "import sys; print(sys.executable)"
```

`conda run` lance la commande dans le contexte d'un env sans avoir besoin de faire `activate`/`deactivate`. Pratique dans des scripts ou des commandes inline (et c'est précisément ce qu'on utilise pour enregistrer un kernel, section suivante).

### Conda vs pip dans un même env

`pip install` fonctionne aussi à l'intérieur d'un env conda (et installe dans cet env). Règle de bonnes pratiques :

1. **Toujours essayer `conda install` en premier** (cohérence ABI avec les autres paquets conda)
2. **Tomber sur `pip install`** seulement si le paquet n'existe pas sur conda-forge ou un autre channel
3. **Ne jamais mélanger sans raison** — mélanger conda et pip dans le même env peut casser les dépendances binaires

In [ ]:
# Détails complets : env actif, chemins, channels, plateforme.
# Très utile pour diagnostiquer "pourquoi mon env ne se comporte pas comme prévu".
!conda info

## 5. Référence pratique : kernels Jupyter

### Enregistrer un env conda comme kernel

Pour que JupyterLab voie ton env conda dans son menu de sélection :

```bash
# 1. Installer ipykernel dans l'env cible (prérequis)
conda install -n mon_env ipykernel -y

# 2. Enregistrer l'env comme kernel
conda run -n mon_env python -m ipykernel install --user \
    --name mon_env \
    --display-name "Python (mon_env)"
```

**Décomposition de la commande d'enregistrement :**

- `conda run -n mon_env python` : exécute le `python` de `mon_env` (sans avoir besoin de `conda activate`)
- `-m ipykernel install` : exécute le module `ipykernel`, sous-commande `install`
- `--user` : installe pour l'utilisateur courant uniquement (pas system-wide)
- `--name mon_env` : nom interne du kernel (utilisé par Jupyter en interne, doit être unique)
- `--display-name "..."` : nom affiché dans le menu de sélection (peut contenir des espaces)

### Inspection et nettoyage

| Commande | Effet |
|---|---|
| `jupyter kernelspec list` | Liste tous les kernels enregistrés avec leur chemin |
| `jupyter kernelspec remove mon_env` | Supprime l'enregistrement du kernel (l'env conda reste intact) |

### Où sont stockés les kernelspecs

| Plateforme | Chemin |
|---|---|
| GX10 (Ubuntu) | `~/.local/share/jupyter/kernels/` |
| Windows | `C:\Users\benoi\AppData\Roaming\jupyter\kernels\` |

Chaque kernel y a un sous-dossier avec un fichier `kernel.json`. Voir la section suivante pour son contenu.

### Vérifier que VSCode voit le nouveau kernel

Sur Windows, après `ipykernel install`, VSCode peut mettre du temps à détecter le nouveau kernel. Forcer le rafraîchissement :

1. `Ctrl+Shift+P` → taper `Jupyter: Restart Kernel`, OU
2. Fermer/rouvrir le notebook, OU
3. Recharger la fenêtre : `Ctrl+Shift+P` → `Developer: Reload Window`

In [ ]:
# Liste les kernels enregistrés (visibles dans le menu de sélection de JupyterLab/VSCode).
# Si "ai_learning" n'apparaît pas ici, c'est que l'enregistrement n'a pas été fait.
!jupyter kernelspec list

## 6. Qu'est-ce qu'un kernel ?

Un **kernel Jupyter** est un processus qui exécute le code des cellules. Quand tu fais Shift+Enter sur une cellule code, JupyterLab :

1. Envoie le code source de la cellule au kernel (via un protocole ZeroMQ)
2. Le kernel exécute le code dans son propre processus Python
3. Le kernel renvoie le résultat (stdout, stderr, valeurs, exceptions, figures...)
4. JupyterLab affiche le résultat sous la cellule

**Analogie LabVIEW :** le kernel est l'équivalent du **Run-Time Engine** qui exécute ton VI. JupyterLab serait l'éditeur (LabVIEW IDE). Tu peux fermer le notebook (l'éditeur) sans tuer le kernel (le runtime), et y revenir plus tard pour récupérer l'état des variables.

**Analogie C/C++ :** le kernel est comme un debugger attaché à un processus. Tu envoies des commandes (du code) au processus via le debugger, et il les exécute en gardant son état entre deux commandes (les variables persistent d'une cellule à l'autre).

### Pourquoi cette architecture ?

La séparation kernel ↔ frontend permet :

- **Plusieurs frontends pour un même kernel** : JupyterLab, VSCode, et un terminal Jupyter peuvent tous se connecter au même kernel et voir les mêmes variables.
- **Kernels distants** : ton frontend peut tourner sur ton laptop et le kernel sur le GX10 via SSH. C'est exactement ce que tu fais en accédant au JupyterLab GX10 depuis ton navigateur Windows.
- **Kernels non-Python** : il existe des kernels R, Julia, Bash, C++ (xeus-cling), etc. JupyterLab ne sait rien de Python en soi — il parle juste à un kernel via le protocole ZeroMQ.

### États possibles d'un kernel

| État | Indicateur visuel | Signification |
|---|---|---|
| Idle (◯) | rond vide | Kernel prêt à recevoir du code |
| Busy (●) | rond plein | Kernel exécute une cellule |
| Disconnected | symbole barré | Le frontend ne trouve plus le kernel |
| Dead | erreur explicite | Le processus kernel a crashé |

## 7. `ipykernel` et le kernelspec

### Le rôle d'`ipykernel`

`ipykernel` est le paquet Python qui **implémente le protocole de kernel Jupyter pour Python**. Sans lui, ton env conda peut avoir Python mais ne peut pas servir de kernel à Jupyter.

Quand tu fais `python -m ipykernel install ...`, ipykernel crée un fichier de configuration (le **kernelspec**) qui dit à Jupyter : "voici un kernel Python disponible, et voici la commande pour le lancer".

### Le fichier `kernel.json`

Sur Windows, l'env `ai_learning` a généré ce fichier dans `C:\Users\benoi\AppData\Roaming\jupyter\kernels\ai_learning\kernel.json` :

```json
{
  "argv": [
    "C:\\Users\\benoi\\.conda\\envs\\ai_learning\\python.exe",
    "-m",
    "ipykernel_launcher",
    "-f",
    "{connection_file}"
  ],
  "display_name": "Python (ai_learning)",
  "language": "python"
}
```

**Décodage ligne par ligne :**

- `argv` : la commande exacte que Jupyter exécute pour démarrer ce kernel. Le `{connection_file}` est un placeholder que Jupyter remplace au lancement par un fichier temporaire contenant les ports ZeroMQ à utiliser.
- Le **premier élément** du `argv` (`python.exe`) est ce qui détermine quel Python est utilisé — donc quel env conda. C'est ce chemin que tu vois quand tu fais `print(sys.executable)` dans une cellule.
- `display_name` : ce que tu vois dans le menu de sélection.
- `language` : pour la coloration syntaxique et les hints.

### Analogie ABI / DLL search

Pense au kernelspec comme à un fichier `.pc` (pkg-config) ou à une entrée du DLL search path : c'est une **petite couche de méta-données** qui dit à un outil (Jupyter) comment trouver et lancer le bon binaire (`python.exe`). Le kernel n'est pas un binaire spécial — c'est juste **Python + ipykernel + un chemin enregistré dans un JSON**.

### Conséquence pratique

Si tu déplaces ton env conda (ou si tu le supprimes), le kernelspec devient cassé : Jupyter le verra encore dans le menu mais échouera au lancement avec "executable not found". 

**La règle :** quand tu supprimes un env, n'oublie pas de supprimer aussi son kernelspec avec `jupyter kernelspec remove <name>`.

In [ ]:
# Lit le kernel.json du kernel "ai_learning" et l'affiche formaté.
# Le chemin diffère entre les plateformes — on le détecte automatiquement.
from pathlib import Path
import json
import os
import sys

if sys.platform == "win32":
    kernels_dir = Path(os.environ["APPDATA"]) / "jupyter" / "kernels"
else:
    kernels_dir = Path.home() / ".local" / "share" / "jupyter" / "kernels"

kernel_file = kernels_dir / "ai_learning" / "kernel.json"
print(f"Lecture de : {kernel_file}\n")
print(json.dumps(json.loads(kernel_file.read_text(encoding="utf-8")), indent=2))

## 8. Les trois sources de "Python actif"

C'est LE point qui cause le plus de confusion : à un moment donné, **plusieurs Python peuvent coexister** sur ta machine, et lequel est "actif" dépend du contexte.

### Source 1 — l'env conda actif dans le shell

Le prompt PowerShell `(ai_learning) PS C:\Users\benoi>` indique que **dans ce shell**, `python` lancera l'interpréteur de `ai_learning`. C'est l'env qui est ajouté en tête du `PATH` par `conda activate`.

**Pour vérifier dans le shell :**

```powershell
python -c "import sys; print(sys.executable)"
```

### Source 2 — le kernel sélectionné dans le notebook

C'est **complètement indépendant** du shell. Un notebook ouvert avec le kernel `Python (ai_learning)` exécutera son code avec le `python.exe` pointé par le `kernel.json` correspondant, **peu importe** l'env actif dans ton shell.

**Pour vérifier dans une cellule code :**

```python
import sys
print(sys.executable)
```

### Source 3 — l'interpréteur Python de VSCode (hors notebook)

VSCode garde un Python "global" pour les fichiers `.py` standalone (linting, débogage, exécution via le bouton Run). Ce Python peut être différent du kernel actif du notebook ouvert dans le même VSCode. Visible en bas à droite de la fenêtre : `3.11.X 64-bit ('ai_learning': conda)`.

### Le piège classique

Tu actives `ai_learning` dans PowerShell, tu lances JupyterLab (ou tu ouvres un notebook dans VSCode), mais ton notebook utilise un kernel `Python 3 (ipykernel)` générique qui pointe vers un autre env (souvent `base`). Tu installes un paquet via `pip install` dans le shell — il va dans `ai_learning`. Mais `import` dans le notebook échoue parce que le kernel utilise un autre Python.

**La règle :** dans un notebook, fais toujours confiance à `sys.executable` plutôt qu'au prompt du shell. `sys.executable` est la vérité absolue de quel env tu utilises *à cet instant, dans ce notebook*.

**Analogie pointeurs :** le prompt shell est comme une variable globale qui dit "voici l'env par défaut". Le kernel notebook est comme un pointeur indépendant qui pointe vers son propre env. Les deux peuvent diverger.

In [ ]:
# La vérité absolue sur quel Python tourne ICI, dans CE notebook, MAINTENANT.
# Indépendant du prompt du shell, indépendant du Python "global" de VSCode.
import sys
print(sys.executable)

## 9. Spécificités GX10 (Ubuntu) vs Windows

### Différences dans les cellules Jupyter

| Commande dans une cellule | GX10 | Windows |
|---|---|---|
| `!conda env list` | échec (`conda: command not found`) | OK |
| `!/home/bbrisson/miniconda3/bin/conda env list` | OK (chemin absolu) | n/a |
| `!python -c "..."` | OK (utilise le python du kernel) | OK |
| `%%bash` + commandes conda | échec (nouveau shell sans conda init) | n/a |
| `!git status` | OK | OK |

**Pourquoi `!conda` échoue sur le GX10 :** JupyterLab tourne dans son propre venv Python 3.12, lancé par le dashboard GX10. Ce processus n'a pas exécuté `~/.bashrc`, donc l'init conda n'a pas modifié son `PATH`. Quand tu fais `!conda`, Jupyter lance un sous-shell qui hérite du `PATH` du processus parent — sans conda.

**Pourquoi ça marche sur Windows :** ton install Windows a fait `conda init` pour PowerShell ET pour `cmd.exe`. Le `!` dans Jupyter sur Windows utilise `cmd.exe` par défaut, qui a conda dans son `PATH`.

### `conda activate` dans les cellules

| | GX10 | Windows |
|---|---|---|
| `!conda activate ai_learning` | ne persiste pas | ne persiste pas |

`conda activate` modifie l'environnement du shell. Mais chaque cellule `!` lance un **nouveau sous-processus** dont l'environnement est jeté après. Pour exécuter quelque chose dans un env sans l'activer, utiliser `conda run -n nom_env <commande>`.

### Modes d'accès à Jupyter

| | GX10 | Windows |
|---|---|---|
| Démarrage | Dashboard GX10 → port 11002 | `jupyter lab` dans PowerShell → port 8888, ou natif VSCode |
| Accès | Navigateur via SSH tunnel (ou config exposée) | Navigateur local (`http://localhost:8888`) ou natif VSCode |
| Working directory | `/home/bbrisson` (configuré dans le dashboard) | Dossier où tu lances `jupyter lab`, ou racine du workspace VSCode |

### Pour commiter depuis un notebook

| | GX10 | Windows |
|---|---|---|
| `!git commit -m "..."` | fonctionne mais peu fiable (escape, encoding) | idem |
| Recommandation | utiliser le terminal SSH | utiliser PowerShell local |

Les opérations git ponctuelles (`!git status`, `!git log`) sont OK dans les cellules. Les commits, eux, gagnent à être faits dans un vrai terminal pour éviter les problèmes d'encoding des messages.

## 10. Vérification de l'environnement

Les cellules ci-dessous vérifient que tout est cohérent : le kernel sélectionné pointe bien vers l'env `ai_learning`, et `ipykernel` y est correctement installé.

**À exécuter avec le kernel `Python (ai_learning)`.**

Les sorties attendues dépendent de la plateforme :

- **Sur GX10** : exécutable du style `/home/bbrisson/miniconda3/envs/ai_learning/bin/python`
- **Sur Windows** : exécutable du style `C:\Users\benoi\.conda\envs\ai_learning\python.exe`

Dans les deux cas, le chemin doit contenir `ai_learning` — sinon c'est qu'un autre kernel est sélectionné (voir la dernière section pour corriger ça).

In [ ]:
import sys
print(f"Python : {sys.version}")

In [ ]:
print(f"Exécutable : {sys.executable}")

In [ ]:
import ipykernel
print(f"ipykernel : {ipykernel.__version__}")

ipykernel : 7.2.0


### Ce que tu dois voir

- Python 3.11.x
- L'exécutable contient `ai_learning` dans son chemin
  - GX10 : `/home/bbrisson/miniconda3/envs/ai_learning/bin/python`
  - Windows : `C:\Users\benoi\.conda\envs\ai_learning\python.exe`
- `ipykernel` version 7.x

Si l'exécutable ne contient **pas** `ai_learning`, c'est qu'un mauvais kernel est sélectionné. Voici comment corriger :

1. Clique sur le **nom du kernel** en haut à droite du notebook (ex: `Python 3 (ipykernel)`)
2. Un menu déroulant apparaît avec la section **Start python Kernel**
3. Sélectionne **Python (ai_learning)**
4. Ré-exécute les cellules

---

### Comprendre le menu de sélection de kernel

Le menu propose trois catégories :

| Option | Quand l'utiliser |
|---|---|
| **Start python Kernel** | Aucun kernel actif — on en démarre un nouveau |
| **Use No Kernel** | Notebook de pure documentation, sans exécution de code |
| **Connect to Existing python Kernel** | Un kernel tourne déjà — on s'y connecte |

**Pourquoi "Connect to Existing" est surligné quand le notebook est déjà ouvert ?**

C'est contre-intuitif, mais logique : JupyterLab fait la distinction entre le **kernel** (le moteur Python) et la **connexion** (le lien entre le notebook et ce kernel).

Le menu répond à la question *"comment veux-tu te connecter ?"*, pas *"quel kernel utilises-tu ?"*.

**Analogie avec la POO :** démarrer un kernel, c'est comme instancier une classe :

```python
kernel = PythonKernel(env="ai_learning")  # un objet est créé en mémoire
```

L'identifiant `08c4c518` visible dans le menu est l'équivalent de l'adresse mémoire de cette instance — il la rend unique. On pourrait avoir deux kernels `ai_learning` actifs simultanément, avec des identifiants différents.

Le notebook se **connecte** à cette instance, comme une variable qui pointe vers un objet :

```python
mon_notebook = kernel  # pointe vers l'instance existante
```

JupyterLab surligne donc **"Connect to Existing"** parce que c'est exactement ce qui se passe : le notebook pointe déjà vers une instance en cours d'exécution.

---

### Pour aller plus loin

- Section **8** ci-dessus si tu veux comprendre pourquoi `sys.executable` peut être différent de ce que ton shell laisse penser.
- Section **9** pour les pièges spécifiques à chaque plateforme dans les cellules Jupyter (`!conda`, `%%bash`, etc.).